# 08 — Sample Statistics & Target Distributions

Quick overview of the dataset used throughout the pipeline:  
target histograms, binary-label imbalance, and a summary table by split.

In [ ]:
# Cell 0.1: Clone repo
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

CSV_PATH = 'data/processed/patches_with_splits.csv'
OUTPUT_FIGURES = Path('outputs/figures')
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df):,} patches  \u2014  splits: {dict(df["split"].value_counts())}')

## 1. Histogram \u2014 `coverage_fraction`

In [ ]:
split_order = ['train', 'val', 'test']
palette = {'train': '#4C72B0', 'val': '#55A868', 'test': '#C44E52'}

fig, ax = plt.subplots(figsize=(8, 4.5))

bins = np.linspace(0, df['coverage_fraction'].max(), 50)

for split in split_order:
    vals = df.loc[df['split'] == split, 'coverage_fraction']
    ax.hist(vals, bins=bins, alpha=0.55, label=f'{split} (n={len(vals):,})',
            color=palette[split], edgecolor='white', linewidth=0.4)

ax.set_xlabel('Coverage Fraction', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Coverage Fraction by Split', fontsize=14, fontweight='bold')
ax.legend(frameon=True, fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

zero_pct = (df['coverage_fraction'] == 0).mean() * 100
ax.annotate(f'{zero_pct:.1f}% of patches\nhave zero coverage',
            xy=(0.01, ax.get_ylim()[1] * 0.6), fontsize=10, fontstyle='italic',
            color='#555555')

plt.tight_layout()
plt.savefig(str(OUTPUT_FIGURES / 'hist_coverage_fraction.png'), dpi=200,
            bbox_inches='tight', facecolor='white')
print(f'Saved: {OUTPUT_FIGURES / "hist_coverage_fraction.png"}')
plt.show()

## 2. Histogram \u2014 `log_mean_tests`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

bins = np.linspace(0, df['log_mean_tests'].max(), 50)

for split in split_order:
    vals = df.loc[df['split'] == split, 'log_mean_tests']
    ax.hist(vals, bins=bins, alpha=0.55, label=f'{split} (n={len(vals):,})',
            color=palette[split], edgecolor='white', linewidth=0.4)

ax.set_xlabel('log(1 + mean tests)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Log Mean Tests by Split', fontsize=14, fontweight='bold')
ax.legend(frameon=True, fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

zero_pct = (df['log_mean_tests'] == 0).mean() * 100
ax.annotate(f'{zero_pct:.1f}% of patches\nhave zero tests',
            xy=(df['log_mean_tests'].max() * 0.55, ax.get_ylim()[1] * 0.6),
            fontsize=10, fontstyle='italic', color='#555555')

plt.tight_layout()
plt.savefig(str(OUTPUT_FIGURES / 'hist_log_mean_tests.png'), dpi=200,
            bbox_inches='tight', facecolor='white')
print(f'Saved: {OUTPUT_FIGURES / "hist_log_mean_tests.png"}')
plt.show()

## 3. Binary Label Imbalance

In [ ]:
df['binary_label'] = (df['coverage_fraction'] > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'width_ratios': [1.3, 1]})

# \u2500\u2500 (A) Stacked bar by split \u2500\u2500
ax = axes[0]
counts = df.groupby(['split', 'binary_label']).size().unstack(fill_value=0)
counts = counts.reindex(split_order)

bar_colors = {0: '#b0bec5', 1: '#43a047'}
x = np.arange(len(split_order))
width = 0.5

bottom = np.zeros(len(split_order))
for label in [0, 1]:
    vals = counts[label].values
    tag = 'Connected' if label == 1 else 'No coverage'
    bars = ax.bar(x, vals, width, bottom=bottom, label=tag,
                  color=bar_colors[label], edgecolor='white', linewidth=0.6)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 100:
            ax.text(x[i], b + v / 2, f'{v:,}', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in split_order], fontsize=11)
ax.set_ylabel('Number of Patches', fontsize=11)
ax.set_title('(a) Label Distribution by Split', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10, frameon=True)
ax.spines[['top', 'right']].set_visible(False)

# \u2500\u2500 (B) Positive rate by split \u2500\u2500
ax = axes[1]
pos_rates = df.groupby('split')['binary_label'].mean().reindex(split_order)

bars = ax.bar(x, pos_rates.values, width,
              color=[palette[s] for s in split_order],
              edgecolor='white', linewidth=0.6)

for i, v in enumerate(pos_rates.values):
    ax.text(x[i], v + 0.01, f'{v:.1%}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in split_order], fontsize=11)
ax.set_ylabel('Positive Rate (has coverage)', fontsize=11)
ax.set_title('(b) Positive Rate by Split', fontsize=13, fontweight='bold')
ax.set_ylim(0, min(pos_rates.max() * 1.35, 1.0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(str(OUTPUT_FIGURES / 'binary_label_imbalance.png'), dpi=200,
            bbox_inches='tight', facecolor='white')
print(f'Saved: {OUTPUT_FIGURES / "binary_label_imbalance.png"}')
plt.show()

## 4. Sample Statistics Table

In [ ]:
rows = []
for split in ['train', 'val', 'test', 'Overall']:
    sub = df if split == 'Overall' else df[df['split'] == split]
    rows.append({
        'Split': split.capitalize(),
        'N': len(sub),
        'Positive rate': f"{sub['binary_label'].mean():.1%}",
        'Coverage fraction mean': f"{sub['coverage_fraction'].mean():.4f}",
        'Coverage fraction std':  f"{sub['coverage_fraction'].std():.4f}",
        'Log-mean tests mean':    f"{sub['log_mean_tests'].mean():.3f}",
        'Log-mean tests std':     f"{sub['log_mean_tests'].std():.3f}",
    })

stats_df = pd.DataFrame(rows)
print(stats_df.to_string(index=False))
stats_df.to_csv('outputs/results/sample_statistics.csv', index=False)
print(f'\nSaved: outputs/results/sample_statistics.csv')

## 5. Spatial Distribution — `log_mean_tests`

All 7,000 patches plotted on an India map, coloured by ground-truth `log_mean_tests`.
The dashed rectangle highlights the Northeast India geographic hold-out (test set).

In [ ]:
!pip install -q cartopy

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize

PATCH_SIZE_DEG = 6720.0 / 111_320.0
VIS_PATCH_DEG  = PATCH_SIZE_DEG * 1.6
half = VIS_PATCH_DEG / 2

PAD = 0.5
EXTENT = [
    df['lon'].min() - PAD, df['lon'].max() + PAD,
    df['lat'].min() - PAD, df['lat'].max() + PAD,
]

# NE India test region bounding box
NE_LON = (88.0, 97.5)
NE_LAT = (21.0, 28.5)

fig, ax = plt.subplots(figsize=(12, 14), subplot_kw={'projection': ccrs.PlateCarree()})
fig.patch.set_facecolor('#1a1a2e')
ax.set_extent(EXTENT, crs=ccrs.PlateCarree())

ax.add_feature(cfeature.OCEAN,   facecolor='#1a1a2e', zorder=0)
ax.add_feature(cfeature.LAND,    facecolor='#2d2d3d', zorder=0)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='#888888', zorder=1)
ax.add_feature(cfeature.STATES,  linewidth=0.25, edgecolor='#555555', zorder=1)

cmap = plt.get_cmap('RdYlGn')
pos = df[df['log_mean_tests'] > 0]
vmax = pos['log_mean_tests'].quantile(0.95)
norm = Normalize(vmin=0, vmax=vmax)

# Zero-value patches in muted gray
zeros = df[df['log_mean_tests'] == 0]
for _, row in zeros.iterrows():
    rect = Rectangle((row['lon'] - half, row['lat'] - half), VIS_PATCH_DEG, VIS_PATCH_DEG,
                      facecolor='#444444', edgecolor='none', alpha=0.5,
                      transform=ccrs.PlateCarree(), zorder=2)
    ax.add_patch(rect)

# Positive-value patches coloured by log_mean_tests
for _, row in pos.iterrows():
    color = cmap(norm(row['log_mean_tests']))
    rect = Rectangle((row['lon'] - half, row['lat'] - half), VIS_PATCH_DEG, VIS_PATCH_DEG,
                      facecolor=color, edgecolor='none', alpha=0.85,
                      transform=ccrs.PlateCarree(), zorder=3)
    ax.add_patch(rect)

# NE India test region highlight
ne_rect = Rectangle((NE_LON[0], NE_LAT[0]), NE_LON[1] - NE_LON[0], NE_LAT[1] - NE_LAT[0],
                     facecolor='none', edgecolor='#ff6b6b', linewidth=2, linestyle='--',
                     transform=ccrs.PlateCarree(), zorder=4)
ax.add_patch(ne_rect)
ax.text(NE_LON[1] + 0.3, (NE_LAT[0] + NE_LAT[1]) / 2, 'NE India\nTest Set',
        fontsize=11, fontweight='bold', color='#ff6b6b', va='center',
        transform=ccrs.PlateCarree())

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink=0.6)
cbar.set_label('log(1 + mean tests)', fontsize=12, color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=9)

# Gray-patch legend note
ax.text(0.02, 0.02, f'Gray = no coverage ({len(zeros):,} patches)\nColoured = has tests ({len(pos):,} patches)',
        transform=ax.transAxes, fontsize=10, color='#aaaaaa', va='bottom',
        bbox=dict(facecolor='#1a1a2e', edgecolor='#555555', alpha=0.9, boxstyle='round,pad=0.4'))

ax.set_title('Ground-Truth Internet Connectivity Across India\nlog(1 + mean Ookla speed tests per sub-tile)',
             fontsize=15, fontweight='bold', color='white', pad=14)
ax.set_facecolor('#1a1a2e')

plt.tight_layout()
plt.savefig(str(OUTPUT_FIGURES / 'spatial_log_mean_tests_india.png'),
            dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
print(f'Saved: {OUTPUT_FIGURES / "spatial_log_mean_tests_india.png"}')
plt.show()

## 6. Push outputs to GitHub (Colab)

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction-ML"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

subprocess.run(['git', 'add', '-f', 'notebooks/08_sample_statistics.ipynb'], check=True)

for pat in ['*.png', '*.pdf']:
    for p in OUTPUT_FIGURES.glob(pat):
        subprocess.run(['git', 'add', '-f', str(p)], check=True)
        print(f'  Staged: {p}')

subprocess.run(['git', 'add', '-f', 'outputs/figures/spatial_log_mean_tests_india.png'],
               capture_output=True)

for p in Path('outputs/results').glob('sample_statistics.csv'):
    subprocess.run(['git', 'add', '-f', str(p)], check=True)
    print(f'  Staged: {p}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB08: Sample statistics + spatial log_mean_tests India map'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')